# 04 — SimCLR Pre-training (Self-Supervised)

**Goal**: pre-train a ResNet18 backbone using SimCLR contrastive learning
on the *full* dataset (no labels used). The resulting encoder is saved
and used in notebook 05 for supervised fine-tuning.

**Key idea**: given a batch of N images, we generate 2N views via strong
augmentations. NT-Xent loss pushes the two views of the same image together
and all other 2N-2 views apart in the projection space.

Checkpoint → `checkpoints/simclr_backbone.pth`  
Figures    → `reports/figures/04_simclr_pretrain/`  
CSV        → `reports/04_simclr_pretrain_history.csv`

## 0. Setup

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (
    (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT / 'datasets').exists()
):
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Cannot locate project root')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as T
from torch.utils.data import DataLoader

# All SimCLR classes exported from src.ssl
from src.ssl import (
    get_simclr_transform,
    UnlabeledPairDataset,
    SimCLR,
    NTXentLoss,
)
from src.utils import fig_path, save_figure, save_results_csv

DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
REPORTS_DIR = str(PROJECT_ROOT / 'reports')
NB          = '04_simclr_pretrain'
CKPT_DIR    = PROJECT_ROOT / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}')
%matplotlib inline

## 1. SimCLR augmentation pipeline

Following Chen et al. (ICML 2020): crop + flip + color jitter + grayscale + blur.
**No gaussian_noise** — from nb03 we know it destroys the discriminative signal.

In [ ]:
IMAGE_SIZE       = 96
SIMCLR_TRANSFORM = get_simclr_transform(IMAGE_SIZE)
print('SimCLR transform:')
print(SIMCLR_TRANSFORM)

### 1b. Visual preview of SimCLR augmentation

Show the same image under 8 different stochastic samples of the SimCLR pipeline.
These are the kinds of *positive pairs* the contrastive loss will see.

In [ ]:
data_root = PROJECT_ROOT / 'datasets' / 'raw'
all_imgs  = sorted(data_root.rglob('*.jpg')) + sorted(data_root.rglob('*.png'))
if not all_imgs:
    raise FileNotFoundError(f'No images found in {data_root}')
base_img = Image.open(all_imgs[len(all_imgs) // 2]).convert('RGB').resize((224, 224))

n_samples = 8
fig, axes = plt.subplots(1, n_samples + 1, figsize=(3 * (n_samples + 1), 3.5))

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

axes[0].imshow(base_img)
axes[0].set_title('original', fontsize=8, fontweight='bold')
axes[0].axis('off')

for k, ax in enumerate(axes[1:]):
    t = SIMCLR_TRANSFORM(base_img)
    t_disp = (t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(t_disp)
    ax.set_title(f'view {k+1}', fontsize=7)
    ax.axis('off')

fig.suptitle('SimCLR augmentation — 8 stochastic views of the same image (= 4 positive pairs)', fontsize=10)
fig.tight_layout()
save_figure(fig, fig_path(REPORTS_DIR, NB, 'simclr_aug_preview.png'), show=True)

## 2. Dataset & DataLoader

In [ ]:
BATCH_SIZE = 128
NUM_WORKERS = 2 # adjust based on your CPU cores (or gpu if using CUDA with pinned memory)

dataset = UnlabeledPairDataset(PROJECT_ROOT / 'datasets' / 'raw', SIMCLR_TRANSFORM)
loader  = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == 'cuda'),  # only pin when GPU is present
    drop_last=True,
)
print(f'Batches per epoch: {len(loader)}')

## 3. Model + loss

In [ ]:
PROJ_DIM  = 128
model     = SimCLR(proj_dim=PROJ_DIM).to(DEVICE)
criterion = NTXentLoss(temperature=0.5)

total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_params:,}')
print(f'  encoder:   {sum(p.numel() for p in model.encoder.parameters()):,}')
print(f'  projector: {sum(p.numel() for p in model.projector.parameters()):,}')

## 4. NT-Xent loss (reminder)

For a batch of N images (2N views), the loss for positive pair (i, j) is:

$$\ell_{i,j} = -\log \frac{\exp(\text{sim}(z_i, z_j)/\tau)}{\sum_{k \neq i} \exp(\text{sim}(z_i, z_k)/\tau)}$$

## 5. Training config

In [ ]:
LR           = 3e-4
NUM_EPOCHS   = 100     # increase to 100+ for a serious run
WEIGHT_DECAY = 1e-4

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print(f'Training for {NUM_EPOCHS} epochs, batch={BATCH_SIZE}, lr={LR}, tau=0.5')

## 6. Training loop

In [ ]:
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0

    for x_i, x_j in loader:
        x_i = x_i.to(DEVICE, non_blocking=True)
        x_j = x_j.to(DEVICE, non_blocking=True)

        _, z_i = model(x_i)
        _, z_j = model(x_j)

        loss = criterion(z_i, z_j)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(loader)
    scheduler.step()
    lr_now = scheduler.get_last_lr()[0]
    history.append({'epoch': epoch, 'loss': avg_loss, 'lr': lr_now})
    print(f'Epoch {epoch:3d}/{NUM_EPOCHS}  loss={avg_loss:.4f}  lr={lr_now:.6f}')

print('\nPre-training complete.')

## 7. Save backbone checkpoint

We save **only the encoder** (`model.encoder.state_dict()`), not the projection head.
Notebook 05 will load this and attach a linear classifier on top.

In [ ]:
ckpt_path = CKPT_DIR / 'simclr_backbone.pth'
torch.save({
    'encoder_state_dict': model.encoder.state_dict(),
    'proj_dim':           PROJ_DIM,
    'feat_dim':           model.feat_dim,
    'num_epochs':         NUM_EPOCHS,
    'batch_size':         BATCH_SIZE,
    'temperature':        0.5,
    'history':            history,
}, ckpt_path)
print(f'Backbone saved → {ckpt_path}')

## 8. Loss curve

In [ ]:
epochs_list = [h['epoch'] for h in history]
losses      = [h['loss']  for h in history]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs_list, losses, 'o-', color='steelblue', linewidth=1.5, markersize=4)
ax.set_xlabel('Epoch')
ax.set_ylabel('NT-Xent Loss')
ax.set_title('SimCLR pre-training loss curve')
ax.grid(alpha=0.3)
fig.tight_layout()
save_figure(fig, fig_path(REPORTS_DIR, NB, 'pretrain_loss.png'), show=True)

## 9. Feature quality — linear probe (quick sanity check)

Freeze the encoder, train a single linear layer on the labeled train split,
evaluate on the test split. Standard SSL evaluation protocol.

> Full fine-tuning is in notebook 05.

In [ ]:
import torchvision.datasets as dsets
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

eval_transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(IMAGE_SIZE),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

labeled_ds = dsets.ImageFolder(str(PROJECT_ROOT / 'datasets' / 'raw'), transform=eval_transform)
g = torch.Generator().manual_seed(42)
n_train = int(0.7 * len(labeled_ds))
n_test  = len(labeled_ds) - n_train
train_ds, test_ds = torch.utils.data.random_split(labeled_ds, [n_train, n_test], generator=g)

# num_workers=0 is safe here: no custom Dataset class, standard ImageFolder
train_loader_lp = DataLoader(train_ds, batch_size=256, shuffle=False, num_workers=0)
test_loader_lp  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=0)

model.eval()

def extract_features(loader):
    feats, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            h, _ = model(x.to(DEVICE))
            feats.append(h.cpu().numpy())
            labels.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labels)

print('Extracting train features...')
X_train, y_train = extract_features(train_loader_lp)
print('Extracting test  features...')
X_test,  y_test  = extract_features(test_loader_lp)

clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
clf.fit(X_train, y_train)

y_pred_lp = clf.predict(X_test)
y_prob_lp = clf.predict_proba(X_test)[:, 1]

acc_lp = accuracy_score(y_test, y_pred_lp)
auc_lp = roc_auc_score(y_test, y_prob_lp)
f1_lp  = f1_score(y_test, y_pred_lp)

print(f'\nLinear probe results (frozen SimCLR backbone):')
print(f'  Accuracy : {acc_lp:.4f}')
print(f'  AUC      : {auc_lp:.4f}')
print(f'  F1       : {f1_lp:.4f}')
print()
print('Reference (supervised ResNet18 weak, nb02): Acc=0.8356  AUC=0.8928  F1=0.8421')
print('If linear probe << supervised, full fine-tuning in nb05 is needed.')

## 10. Save results to CSV

In [ ]:
save_results_csv(history, str(Path(REPORTS_DIR) / '04_simclr_pretrain_history.csv'))

probe_summary = [{
    'model':           'SimCLR_linear_probe',
    'accuracy':        round(acc_lp, 4),
    'auc':             round(auc_lp, 4),
    'f1':              round(f1_lp,  4),
    'epochs_pretrain': NUM_EPOCHS,
    'batch_size':      BATCH_SIZE,
    'proj_dim':        PROJ_DIM,
}]
save_results_csv(probe_summary, str(Path(REPORTS_DIR) / '04_simclr_probe_results.csv'))

print(f'Done. Checkpoint → checkpoints/simclr_backbone.pth')
print(f'      Figures   → reports/figures/{NB}/')
print( '      CSV       → reports/04_simclr_pretrain_history.csv')
print( '      CSV       → reports/04_simclr_probe_results.csv')